In [4]:
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
import torch
import os
from sklearn.model_selection import train_test_split

model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

dataset = load_dataset("noorulhasan/Chatbot_QnA")
print(f"Dataset structure: {dataset}")
df = dataset["train"]
print(f"First example: {df[0]}")

train_indices, val_indices = train_test_split(
    range(len(df)), 
    test_size=0.15, 
    random_state=42
)
train_df = df.select(train_indices)
val_df = df.select(val_indices)

def format_qa(example):
    return {
        "text": f"Human: {example['Question']}\nBot: {example['Answer']}<|endoftext|>"
    }

train_df = train_df.map(format_qa)
val_df = val_df.map(format_qa)

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
        return_tensors=None
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    for i, attention_mask in enumerate(tokenized["attention_mask"]):
        labels = tokenized["labels"][i]
        for j, mask in enumerate(attention_mask):
            if mask == 0:
                labels[j] = -100
    
    return tokenized

train_ds = train_df.map(tokenize_function, batched=True, remove_columns=["text"])
val_ds = val_df.map(tokenize_function, batched=True, remove_columns=["text"])

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Training samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

output_dir = "./lora-dialogpt-chatbot"
os.makedirs(output_dir, exist_ok=True)

ModuleNotFoundError: No module named 'datasets'